In [ ]:
import os
from datetime import UTC, datetime, timedelta

import httpx
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()

In [ ]:
@tool
async def query_loki(logql: str, limit: int = 100) -> str:
    """Run a LogQL query against Loki, scoped to the last 15 minutes.
    Loki's instant /query endpoint rejects plain log-stream selectors, so this
    delegates to the range endpoint under a short default window."""
    max_limit = int(os.environ.get("LOKI_MAX_LIMIT", "500"))
    limit = max(1, min(limit, max_limit))
    end = datetime.now(UTC)
    start = end - timedelta(minutes=15)
    loki_url = os.environ.get("LOKI_URL", "http://localhost:3100")
    url = f"{loki_url}/loki/api/v1/query_range"
    headers = {}
    token = os.environ.get("LOKI_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
    params = {
        "query": logql,
        "start": start.isoformat(),
        "end": end.isoformat(),
        "limit": limit,
    }
    async with httpx.AsyncClient(timeout=10.0, headers=headers) as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        return str(response.json())[:4000]

In [ ]:
llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ.get("LLM_API_KEY"),
    temperature=0,
).bind_tools([query_loki])

In [ ]:
SYSTEM = "You are an infrastructure-operations assistant. Use the query_loki tool when relevant, then answer concisely."

transcript: list = []


async def ask(question: str) -> str:
    """Ask one question. Makes at most one query_loki call, then returns the model's answer."""
    messages = [SystemMessage(SYSTEM), HumanMessage(question)]
    reply = await llm.ainvoke(messages)
    messages.append(reply)
    if reply.tool_calls:
        call = reply.tool_calls[0]
        result = await query_loki.ainvoke(call["args"] or {})
        messages.append(ToolMessage(tool_call_id=call["id"], content=str(result)[:4000]))
        reply = await llm.ainvoke(messages)
        messages.append(reply)
    transcript[:] = messages
    return reply.content

In [ ]:
print(await ask('Query loki with `{service="api"} |= "error"` and tell me what it found.'))

In [ ]:
def show_transcript(transcript: list) -> None:
    """Print each message with its tool calls, token usage, and reasoning."""
    for m in transcript:
        label = type(m).__name__
        body = m.content if not isinstance(m, ToolMessage) else f"[tool] {m.content}"
        if not body:
            body = "(tool call only)" if getattr(m, "tool_calls", None) else "(empty)"
        print(f"{label}: {body.replace(chr(10), chr(10) + '  ')}")

        reasoning = m.additional_kwargs.get("reasoning_content")
        if reasoning:
            print(f"  reasoning: {reasoning}")

        for call in getattr(m, "tool_calls", None) or []:
            print(f"  tool_call: {call['name']}({call['args']})")

        usage = getattr(m, "usage_metadata", None)
        if usage:
            print(f"  tokens: in={usage['input_tokens']} out={usage['output_tokens']} total={usage['total_tokens']}")

In [ ]:
show_transcript(transcript)